# MOD-12930 — balanced full suite

Same contenders and axes as the main (imbalanced) suite, but the text query of every
(size, depth) cell is **calibrated** so the SEARCH mirror's p50 matches the VSIM mirror's
(±20%, geometric bisection on the target match count; the achieved ratio is recorded).

**Why balance:** with skewed branches, `hybrid ≈ max + C` and `hybrid ≈ sum + C'` fit the
same measurement (sum − max = the small branch ≈ noise), so neither concurrency nor
overhead is identifiable. Equal branches maximize sum − max, so the workers-0/workers-N
pair separates "branches overlap" from "serial overhead C" and yields C twice, independently.

**Matrix:** size (10K/100K/500K) × depth K/W (10/20, 50/100, 250/500 — giving the
C(WINDOW) curve) × workers (0/4) × fields (none/title+text) × 4 contenders.
A cell whose calibration cannot reach balance is recorded with `balanced=False`.

In [1]:
import json, os
import pandas as pd

# Set REUSE_RESULTS=1 to re-render the tables from a previously saved run
# without re-executing the benchmark.
if os.environ.get('REUSE_RESULTS') and os.path.exists('results_balanced_full.json'):
    d = json.load(open('results_balanced_full.json'))
    results, gates, meta = d['results'], d['gates'], d['meta']
    print('reusing saved results_balanced_full.json')
else:
    import bench_lib as B
    import balanced_lib as BAL
    titles, texts, emb, corpus_max = B.load_data()
    results, gates, meta = BAL.run_balanced_full(titles, texts, emb, corpus_max)
    with open('results_balanced_full.json', 'w') as f:
        json.dump(dict(meta=meta, results=results, gates=gates), f, indent=2, default=str)
    pd.DataFrame(results).to_csv('results_balanced_full.csv', index=False)
    print('saved results_balanced_full.json / .csv')

reusing saved results_balanced_full.json


## Calibration & gates

`balance_ratio` = search p50 / vsim p50 at calibration (1.0 = perfect balance; a cell counts as balanced within ±20%).

In [2]:
pd.DataFrame(gates)

,size,k,window,matches_mean,balance_ratio,gate_linear,gate_rrf
0,10000,10,20,2469.37500,1.13,16/16,16/16
1,10000,50,100,2469.37500,1.14,16/16,16/16
2,10000,250,500,2791.31250,1.11,16/16,16/16
3,100000,10,20,1689.31250,1.03,16/16,16/16
4,100000,50,100,5626.56250,0.96,16/16,16/16
5,100000,250,500,12625.46875,0.85,16/16,16/16
6,500000,10,20,1336.15625,1.04,16/16,16/16
7,500000,50,100,2781.40625,1.01,16/16,16/16
8,500000,250,500,4827.93750,0.85,16/16,16/16


## p50 latency (ms)

In [3]:
df = pd.DataFrame(results)
pivot = df.pivot_table(index=['size', 'window', 'workers', 'fields'], columns='contender',
                       values='p50_ms', sort=False).round(2)
pivot[['hybrid_linear', 'hybrid_rrf', 'search_branch', 'vsim_branch']]

contender                         hybrid_linear  hybrid_rrf  search_branch  \
size   window workers fields                                                 
10000  20     0       none                 0.46        0.44           0.27   
                      title+text           0.51        0.53           0.34   
              4       none                 0.44        0.44           0.29   
                      title+text           0.46        0.48           0.34   
       100    0       none                 0.90        0.88           0.37   
                      title+text           0.98        0.99           0.44   
              4       none                 0.72        0.69           0.37   
                      title+text           0.82        0.80           0.43   
       500    0       none                 3.07        3.04           0.87   
                      title+text           3.44        3.60           0.91   
              4       none                 2.33        2.32           0.94   
                      title+text           2.56        2.58           0.88   
100000 20     0       none                 0.53        0.53           0.23   
                      title+text           0.57        0.55           0.37   
              4       none                 0.39        0.41           0.26   
                      title+text           0.44        0.50           0.34   
       100    0       none                 1.24        1.27           0.45   
                      title+text           1.38        1.36           0.45   
              4       none                 0.86        0.90           0.41   
                      title+text           0.98        1.00           0.53   
       500    0       none                 4.51        4.55           1.27   
                      title+text           5.15        5.32           1.45   
              4       none                 3.06        3.00           1.30   
                      title+text           3.46        3.47           1.35   
500000 20     0       none                 0.47        0.47           0.27   
                      title+text           0.49        0.49           0.34   
              4       none                 0.36        0.43           0.27   
                      title+text           0.45        0.47           0.31   
       100    0       none                 1.38        1.34           0.53   
                      title+text           1.44        1.35           0.55   
              4       none                 0.90        0.90           0.48   
                      title+text           1.04        1.04           0.53   
       500    0       none                 4.71        4.66           1.37   
                      title+text           5.42        5.40           1.48   
              4       none                 3.06        3.07           1.31   
                      title+text           3.55        3.58           1.35   

contender                         vsim_branch  
size   window workers fields                   
10000  20     0       none               0.15  
                      title+text         0.24  
              4       none               0.16  
                      title+text         0.22  
       100    0       none               0.23  
                      title+text         0.31  
              4       none               0.26  
                      title+text         0.33  
       500    0       none               0.81  
                      title+text         0.94  
              4       none               0.93  
                      title+text         0.89  
100000 20     0       none               0.19  
                      title+text         0.25  
              4       none               0.20  
                      title+text         0.32  
       100    0       none               0.43  
                      title+text         0.52  
              4       none               0.43  
                      title

## C — the serial overhead, and the C(WINDOW) curve

fields=none. `C_w0 = hybrid − (search+vsim)`, `C_w6 = hybrid − max(search,vsim)`
(mean latency). If depletion truly overlaps and C is genuinely serial, the two agree.
`C/max` says how the overhead compares to running one entire branch.

In [4]:
m = df[df.fields == 'none'].pivot_table(index=['size', 'window', 'workers'],
        columns='contender', values='mean_ms', sort=False)
mx = m[['search_branch', 'vsim_branch']].max(axis=1)
sm = m[['search_branch', 'vsim_branch']].sum(axis=1)
c = pd.DataFrame({'hybrid': m['hybrid_linear'], 'max_branch': mx, 'sum_branch': sm})
c['C_ms'] = (c['hybrid'] - c['sum_branch']).where(
    c.index.get_level_values('workers') == 0, c['hybrid'] - c['max_branch'])
c['C_pct_of_max'] = (100 * c['C_ms'] / c['max_branch'])
c['gain_hint'] = None
c.round(2)

hybrid  max_branch  sum_branch  C_ms  C_pct_of_max  \
size   window workers                                                       
10000  20     0          0.48        0.28        0.43  0.05         16.35   
              4          0.47        0.30        0.47  0.17         55.34   
       100    0          0.92        0.40        0.65  0.28         68.97   
              4          0.73        0.40        0.67  0.33         83.22   
       500    0          3.14        0.89        1.72  1.42        159.43   
              4          2.36        0.97        1.90  1.39        143.94   
100000 20     0          0.55        0.25        0.45  0.10         39.19   
              4          0.41        0.27        0.48  0.14         51.65   
       100    0          1.29        0.55        0.98  0.31         56.03   
              4          0.94        0.52        0.95  0.41         78.93   
       500    0          4.67        1.42        2.82  1.85        129.99   
              4          3.23        1.46        2.90  1.77        121.08   
500000 20     0          0.51        0.29        0.51  0.00          1.47   
              4          0.40        0.30        0.57  0.11         36.69   
       100    0          1.40        0.57        1.06  0.35         61.04   
              4          0.94        0.52        1.00  0.42         80.91   
       500    0          4.77        1.66        3.07  1.70        102.73   
              4          3.15        1.61        2.97  1.53         95.06   

                      gain_hint  
size   window workers            
10000  20     0            None  
              4            None  
       100    0            None  
              4            None  
       500    0            None  
              4            None  
100000 20     0            None  
              4            None  
       100    0            None  
              4            None  
       500    0            None  
              4            None  
500000 20     0            None  
              4            None  
       100    0            None  
              4            None  
       500    0            None  
              4            None

### Parallelism gain per depth (`p50(w0)/p50(w_max)`, fields=none)

In [5]:
wmax = df['workers'].max()
p = df[df.fields == 'none'].pivot_table(index=['size', 'window', 'contender'],
        columns='workers', values='p50_ms', sort=False)
p.columns = [f'w{c}' for c in p.columns]
p['gain'] = (p['w0'] / p[f'w{wmax}']).round(2)
p.round(2)

w0    w4  gain
size   window contender                      
10000  20     hybrid_linear  0.46  0.44  1.04
              hybrid_rrf     0.44  0.44  1.00
              search_branch  0.27  0.29  0.94
              vsim_branch    0.15  0.16  0.90
       100    hybrid_linear  0.90  0.72  1.26
              hybrid_rrf     0.88  0.69  1.28
              search_branch  0.37  0.37  0.99
              vsim_branch    0.23  0.26  0.90
       500    hybrid_linear  3.07  2.33  1.32
              hybrid_rrf     3.04  2.32  1.31
              search_branch  0.87  0.94  0.92
              vsim_branch    0.81  0.93  0.87
100000 20     hybrid_linear  0.53  0.39  1.34
              hybrid_rrf     0.53  0.41  1.31
              search_branch  0.23  0.26  0.88
              vsim_branch    0.19  0.20  0.95
       100    hybrid_linear  1.24  0.86  1.45
              hybrid_rrf     1.27  0.90  1.42
              search_branch  0.45  0.41  1.11
              vsim_branch    0.43  0.43  1.01
       500    hybrid_linear  4.51  3.06  1.47
              hybrid_rrf     4.55  3.00  1.52
              search_branch  1.27  1.30  0.98
              vsim_branch    1.42  1.46  0.97
500000 20     hybrid_linear  0.47  0.36  1.28
              hybrid_rrf     0.47  0.43  1.08
              search_branch  0.27  0.27  0.98
              vsim_branch    0.21  0.27  0.79
       100    hybrid_linear  1.38  0.90  1.53
              hybrid_rrf     1.34  0.90  1.49
              search_branch  0.53  0.48  1.11
              vsim_branch    0.49  0.48  1.02
       500    hybrid_linear  4.71  3.06  1.54
              hybrid_rrf     4.66  3.07  1.52
              search_branch  1.37  1.31  1.05
              vsim_branch    1.70  1.65  1.03